# 青木まりこ現象｜分析① 書店 vs 図書館の滞在時間

**研究問い**：書店の滞在時間は図書館と違うのか？

- 原資料：松田千春（2003）「情報探索におけるブラウジング行動：図書館と書店における行動観察を基にして」『Library and Information Science』49, pp.1–31（表5–8）
- PDF：https://www.libinformsci.com/10.46895/lis.49.1/data/index.pdf
- 2002年観察、図書館2館・書店2店、それぞれ10人（合計40人）
- 原論文の表5〜8から転記した個票をこのノートに埋め込み、**Colabでそのまま実行可能**にしています。
- **便意データは含まれません。** この比較だけで青木まりこ現象の原因は分かりません。

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, kruskal

# Colabで日本語フォントが使用できなければグラフラベルのみ英語表示にする
try:
    import japanize_matplotlib
    JP = True
except ImportError:
    JP = False
    print('Japanese font helper is not installed. Charts will use English labels.')

from pathlib import Path

PROJECT_DIR = Path.cwd()
for folder in (PROJECT_DIR, *PROJECT_DIR.parents):
    if (folder / 'README.md').is_file() and (folder / 'notebooks').is_dir():
        PROJECT_DIR = folder
        break

TABLE_DIR = PROJECT_DIR / 'results' / 'tables'
FIGURE_DIR = PROJECT_DIR / 'results' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


## 1. 観察データを読み込む

松田（2003）の表5〜8から転記した40件のCSVを読み込み、滞在秒から分に換算します。CSVは `data/processed/` を正本とし、Notebook内に同じデータを重複して持ちません。


In [ ]:
# データの正本: data/processed/matsuda_40_data_checked.csv
# GitHubからColabで開いた場合は、必要なときだけ公開CSVを取得する。
DATA_PATH = PROJECT_DIR / 'data' / 'processed' / 'matsuda_40_data_checked.csv'
if not DATA_PATH.is_file():
    url = 'https://raw.githubusercontent.com/Roki0i/aoki-mariko-research/main/data/processed/matsuda_40_data_checked.csv'
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(response.content)

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df['場所区分'] = df['施設'].apply(lambda v: '図書館' if str(v).startswith('図書館') else '書店')
df['滞在分'] = df['滞在秒'] / 60
assert len(df) == 40
assert df['施設'].value_counts().to_dict() == {'図書館A': 10, '図書館B': 10, '書店C': 10, '書店D': 10}
assert (pd.to_timedelta(df['滞在時間_hms']).dt.total_seconds() == df['滞在秒']).all(), '転記した時間に不一致があります'


## 2. 記述統計（図書館 vs 書店、4施設別）

In [ ]:
group_order = ['図書館','書店']
facility_order = ['図書館A','図書館B','書店C','書店D']
summary = (df.groupby('場所区分')['滞在分']
           .agg(n='count', 平均='mean', 中央値='median', 標準偏差='std', 最小='min', 最大='max')
           .reindex(group_order).round(2))
by_facility = (df.groupby('施設')['滞在分']
               .agg(n='count', 平均='mean', 中央値='median', 最小='min', 最大='max')
               .reindex(facility_order).round(2))
print('分類別（分）'); display(summary)
print('施設別（分）'); display(by_facility)


## 3. 箱ひげ図＋個票の表示

平均ではなく、個々人の分布や極端な長時間滞在も確認します。

In [ ]:
labels = group_order if JP else ['Library','Bookstore']
values = [df.loc[df['場所区分']==v,'滞在分'].to_numpy() for v in group_order]
fig, ax = plt.subplots(figsize=(8,5))
ax.boxplot(values, tick_labels=labels, showfliers=False, whis=(0,100))
for i, a in enumerate(values, 1):
    ax.scatter(i + np.random.default_rng(50+i).normal(0,.07,len(a)), a, alpha=.65, s=26)
ax.set_ylabel('滞在時間（分）' if JP else 'Dwell time (minutes)')
ax.set_title('図書館 vs 書店（各20人）' if JP else 'Libraries vs Bookstores (n=20 each)')
ax.grid(axis='y',alpha=.2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'aoki_dwell_group_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
values4 = [df.loc[df['施設']==v,'滞在分'].to_numpy() for v in facility_order]
labels4 = facility_order if JP else ['Library A','Library B','Bookstore C','Bookstore D']
fig, ax = plt.subplots(figsize=(9,5))
ax.boxplot(values4, tick_labels=labels4, showfliers=False, whis=(0,100))
for i, a in enumerate(values4, 1):
    ax.scatter(i + np.random.default_rng(80+i).normal(0,.06,len(a)), a, alpha=.65, s=26)
ax.set_ylabel('滞在時間（分）' if JP else 'Dwell time (minutes)')
ax.set_title('施設別の滞在時間（各10人）' if JP else 'Dwell time by facility (n=10 each)')
ax.grid(axis='y',alpha=.2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'aoki_dwell_facility_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()


## 4. 探索的な統計検定

- Mann–Whitney U検定：2分類の分布比較（両側）
- Kruskal–Wallis検定：4施設間の比較（参考）
- Cliff's delta：2分類での大小関係の効果量（図書館−書店）

**重要**：施設は2館・2店だけです。同一施設内の観察者に相関があり得るため、個人40人を独立なサンプルとみなすp値は**探索的な参考値**です。

In [ ]:
library = values[0]
bookstore = values[1]
mw = mannwhitneyu(library, bookstore, alternative='two-sided')
kw = kruskal(*values4)
cliff = np.mean(np.sign(library[:,None] - bookstore[None,:]))
print(f'Mann–Whitney U = {mw.statistic:.1f}, p = {mw.pvalue:.4f}')
print(f'Kruskal–Wallis H = {kw.statistic:.3f}, p = {kw.pvalue:.4f}')
print(f"Cliff's delta（図書館−書店）= {cliff:.3f}")
print(f'平均差（図書館−書店）= {library.mean()-bookstore.mean():.2f}分')
print(f'中央値差（図書館−書店）= {np.median(library)-np.median(bookstore):.2f}分')
print(f'図書館Aの90分超滞在者 = {np.sum(values4[0]>90)}人')


## 5. 結果の解釈・限界

**観察結果**：図書館の平均時間は長い一方、中央値の差は小さい。Mann–Whitney U検定では5%水準で有意差が検出されない。

**限界**：
1. 便意を記録していない（滞在時間と便意を結び付けられない）。
2. 図書館・書店が各2施設で、施設差を十分にコントロールできない。
3. 原論文では**書店Cで1時間前後滞在した客を追跡し切れず数字に反映されていない**と明記（p.12）。
4. 図書館Aに90分超の3名がいるため平均は外れ値に敏感。
5. 有意差なしは**差がない証明ではない**。

次の段階では、ATUS等の公開個票で大規模な滞在時間分布を再分析します。ただしATUSに書店単独の場所コードはなく、一般の小売店カテゴリとの比較にとどまります。

In [ ]:
# 元データのCSVは上書きせず、集計表と図だけを保存する。
summary.to_csv(TABLE_DIR / 'matsuda_group_summary.csv', encoding='utf-8-sig')
by_facility.to_csv(TABLE_DIR / 'matsuda_facility_summary.csv', encoding='utf-8-sig')
print('分析結果を results/ に保存しました（図表2件・集計表2件）')
